In [29]:
import pandas as pd
import numpy as np


print('hello world')

df = pd.read_csv('data/raw/Employee.csv')

display(df.sample(10))


hello world


,employee_id,age,salary,department,tenure_years,performance_score,left_company
140,EMP0166,43.0,NaN,HR,6.0,2.0,0
84,EMP0114,67.6,74165.15,Engineering,4.9,3.0,0
279,EMP0170,29.0,46362.73,Engineering,3.9,3.0,0
184,EMP0239,30.1,70094.39,Marketing,2.4,3.0,0
31,EMP0285,NaN,57727.99,Marketing,9.7,4.0,0
244,EMP0228,25.1,55721.27,Marketing,3.1,3.0,0
124,EMP0235,63.7,45222.46,Engineering,0.3,5.0,0
219,EMP0268,20.8,39878.71,HR,5.2,4.0,0
222,EMP0241,28.5,67535.38,Engineering,3.0,5.0,1
69,EMP0178,55.4,59668.75,Finance,0.5,5.0,1


In [30]:
print(f"Rows: {df.shape[0]}  Columns: {df.shape[1]}")
display(df.info())

Rows: 303  Columns: 7
<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   employee_id        303 non-null    str    
 1   age                279 non-null    float64
 2   salary             276 non-null    float64
 3   department         303 non-null    str    
 4   tenure_years       303 non-null    float64
 5   performance_score  280 non-null    float64
 6   left_company       303 non-null    int64  
dtypes: float64(4), int64(1), str(2)
memory usage: 16.7 KB


None

In [31]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
null_summary = pd.DataFrame({'nulls': null_counts, 'pct': null_pct})
display(null_summary[null_summary['nulls'] > 0])
display("Null values in each column:", null_counts) 

,nulls,pct
age,24,7.9
salary,27,8.9
performance_score,23,7.6


'Null values in each column:'

employee_id           0
age                  24
salary               27
department            0
tenure_years          0
performance_score    23
left_company          0
dtype: int64

In [41]:
display(df.describe().round(2))

,age,salary,tenure_years,performance_score,left_company
count,279.00,276.00,303.00,280.00,303.00
mean,41.75,91819.19,4.29,3.40,0.15
std,59.45,599227.35,4.24,0.98,0.36
min,-4.00,-500.00,-2.50,1.00,0.00
25%,29.75,44110.68,1.00,3.00,0.00
50%,38.50,55133.78,3.10,3.00,0.00
75%,45.80,64169.79,6.05,4.00,0.00
max,999.00,9999999.00,23.40,5.00,1.00


check if column AGE is missing accross departments or does it have a relations

In [33]:
df['age_missing'] = df['age'].isnull().astype(int)
display(df.groupby('department')['age_missing'].mean().round(2))


department
Engineering    0.04
FINANCE        0.00
Finance        0.10
HR             0.08
Hr             0.00
Marketing      0.12
SALES          0.00
Sales          0.09
engineering    0.14
marketing      0.00
Name: age_missing, dtype: float64

from what i can see the null values are accross the whole department and its just noises. can be filled up or just use imputer on a pipeline for model.

# Check if there is a relation in missing values of salary to department col

In [34]:
df['missing_salary'] = df['salary'].isnull().astype(int)
display(df.groupby('department')['missing_salary'].mean().round(2))

department
Engineering    0.00
FINANCE        0.00
Finance        0.26
HR             0.36
Hr             1.00
Marketing      0.00
SALES          0.00
Sales          0.00
engineering    0.00
marketing      0.00
Name: missing_salary, dtype: float64

for what i can see i can only found 2 departments that has missing values so i feel like there might be a relation on this one
- it is possible that they are not noises and might be a structure on why they are missing
- i believe this needs to be checked and expand research before proceeding to the model

# CHECK IF THE NULL VALUES FROM PERFORMANCE_SCORE HAS RELATIONS TO LEFT COMPANY

In [35]:
df['missing_performance_score'] = df['performance_score'].isnull().astype(int)
display(df.groupby('left_company')['missing_performance_score'].mean().round(2))

left_company
0    0.00
1    0.51
Name: missing_performance_score, dtype: float64

This one is rather obvious for the structure, it is best to assume that the missingness on the performance score is because they left the company. assuming all of the missingness has connections.

# use iqr for numerical testing of outliers

In [60]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return lower, upper

# display(df.columns)
for col in ['age', 'salary', 'tenure_years']:
    low, up = iqr_bounds(df[col])
    outliers = df[(df[col] < low) | (df[col] > up)]
    print(f"\n{col}: {low:.2f} - {up:.2f} are normal values")
    print(f"{col}: {len(outliers)} outliers")
    if len(outliers):
        print(outliers[[col, 'department']].sort_values(col))


age: 5.68 - 69.88 are normal values
age: 6 outliers
       age   department
71    -4.0           HR
149   -0.9  Engineering
0     70.6        Sales
208   84.2        Sales
10   187.0        Sales
267  999.0        Sales

salary: 14022.02 - 94258.45 are normal values
salary: 4 outliers
         salary   department
147     -500.00  engineering
109   101183.21        Sales
94    450000.00           HR
126  9999999.00        Sales

tenure_years: -6.57 - 13.62 are normal values
tenure_years: 15 outliers
     tenure_years   department
209          13.8  Engineering
23           13.9  Engineering
295          13.9    Marketing
237          14.2  Engineering
147          14.5  engineering
72           14.7  Engineering
154          14.9      Finance
174          15.9  Engineering
20           16.5           HR
246          17.3  Engineering
22           17.6    Marketing
166          18.9        Sales
168          21.1        Sales
288          22.4  Engineering
158          23.4        Sales

From what i can see from the iqr output: 
- age column might actually be valid since the numbers have negative values which you might never see this column moreover there are ages that far exceeds the human capabilities but still there are 2 values that needs to be checked

- salary i think is valid, negative values and the other 3 are high earners maybe. they might be the owner of company so needs to check also

- tenure_years column has some invalid output with iqr i'd say. from the output we cann se that there are negative values for the normal distribution which doesn't make sense and the ones it flags as outliers seems to be normal.



Using z - score method to test the outliers

In [50]:
from scipy import stats

for col in ['age', 'salary']:
    z = np.abs(stats.zscore(df[col].dropna()))
    n_out = (z > 3).sum()
    print(f'{col}: {n_out} outliers')

age: 1 outliers
salary: 1 outliers


Z-score only provides 1 outliers for age and salary, i'd say this would be normal since it the value from before has 999 for age and 9999999 salary and would safely assume that those are the value that is outside the mean

## CLEANING

In [ ]:
df_clean = df.copy()


impossible_age = df_clean[(df_clean['age'] < 18) | (df_clean['age'] > 70)] # Assuming 18-70 is the reasonable age range for employees
print(f"Sorted impossible ages:\n{sorted(impossible_age['age'].values.tolist())}")
df_clean['age'] = df_clean['age'].clip(lower=18, upper=70)

Sorted impossible ages:
[-4.0, -0.9, 6.6, 13.7, 14.1, 14.5, 14.6, 15.0, 15.0, 15.6, 16.8, 17.3, 70.6, 84.2, 187.0, 999.0]


In [68]:
extreme_salaries = df_clean[df_clean['salary'] > 500000] # Assuming salaries above $500,000 are extreme outliers
print(f"Extreme salaries:\n{extreme_salaries['salary'].values.tolist()}")
df_clean = df_clean[df_clean['salary'] < 900000]

Extreme salaries:
[]


In [73]:
negative_tenure = df_clean[df_clean['tenure_years'] < 0]
print(f"Negative tenure values:\n{negative_tenure['tenure_years'].values.tolist()}")
df_clean = df_clean[df_clean['tenure_years'] >= 0]



Negative tenure values:
[]


In [75]:
df_clean = df_clean.drop_duplicates(subset=df_clean.columns.difference(['employee_id'])).reset_index(drop=True)
print(f"Number of duplicate rows removed: {len(df) - len(df_clean)}")

print("\n── Final null check ──")
print(df_clean.isnull().sum())
print(f"\nFinal shape: {df_clean.shape}")

df_clean.to_csv('data/processed/employee_clean.csv', index=False)
print("\nSaved: employee_clean.csv")

Number of duplicate rows removed: 31

── Final null check ──
employee_id           0
age                  24
salary                0
department            0
tenure_years          0
performance_score    21
left_company          0
dtype: int64

Final shape: (272, 7)

Saved: employee_clean.csv
